# 01 · Data Exploration — Options Surface Lab

Look at the actual panel: raw LSEG shape → tidy → wide, the sparsity structure, and the
settle-vs-trade gap. This feeds the FR-7 three sentences and the T-7 pull verification
([RUNBOOK](../docs/RUNBOOK.md) §3).

**Discipline (AD-3):** this notebook *consumes* the `options_surface_lab` package — no
transform logic lives here. Anything worth keeping graduates into
`option_surface_utils.py` with a test.

**Kernel:** the `algo` conda env (Python 3.12). `option_pipeline_data.pkl` landed 2026-08-31
(T-7), so this runs on the **real** panel: 296 series × 53 trading days.

**There is no `SETTLE`.** No settlement price is published for US listed equity options — not
by the exchanges, not OPRA, not the OCC (§10b). The wide table's column is `MARK`, a *slot*
filled by `MARK_FIELD_DEFAULT` = `MID_PRICE`, the closing NBBO midpoint (T-32).

In [1]:
import sys
from pathlib import Path

# repo root on sys.path whether the kernel starts in notebooks/ or the repo root
_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import plotly.express as px

from options_surface_lab.option_surface_utils import (
    attach_underlying,
    flatten_lseg_options,
    load_payload,
    pivot_trade_settle,
    summarize_sparsity,
)

payload = load_payload()
print("synthetic:", payload["synthetic"], "| ticker:", payload["ticker"], "| fetched:", payload["fetched_at"])

C:\Users\rjd61\Documents\Duke\development\fintech_535_adv_algo_trading\options_surface_lab\options_surface_lab\option_surface_utils.py:402: RuntimeWarning: option_pipeline_data.pkl not found — using synthetic UUUU-like options so the lab still runs.
  warnings.warn(


synthetic: True | ticker: UUUU | fetched: 2026-08-29 23:40:20


### 1a — Which panel is this, really? (run me first)

Two traps this notebook has already sprung:

- **A `MARK` column is not mark data.** `pivot_trade_settle()` creates the column
  unconditionally — if the pull never returned the mark field you still get a `MARK` column,
  full of `NaN`. Count non-nulls, never `"MARK" in wide.columns`. (The column was called
  `SETTLE` until T-32; there is no settle for these contracts — §10b.)
- **No cache means synthetic.** `load_payload()` silently falls back to a *generated* panel
  when `option_pipeline_data.pkl` is absent. That fallback always has marks and puts, so it
  looks exactly like a successful pull. Check the `SOURCE` line the next cell prints.

### Reading a RIC — calls vs puts

```
UUUU  F  12  26  01100  .U  ^F26
────  ─  ──  ──  ─────  ──  ────
root  │  day yr  strike venue expired-contract suffix
      └─ month letter: A–L = Jan–Dec CALLS,  M–X = Jan–Dec PUTS
```

`F` is the 6th letter → **June call**. A June *put* would be `R` (`UUUUR122601100.U^R26`).
You never have to count letters by hand: `parse_option_ric(ric)["cp"]` returns `'C'`/`'P'`,
and both the tidy and wide tables carry a `cp` column — so `wide["cp"].value_counts()` is the
real answer.

In [ ]:
# Provenance — answers "what am I actually looking at?" for whatever is loaded above.
from options_surface_lab.option_surface_utils import parse_option_ric

_o = payload["options"]
_syn = payload.get("synthetic")
print("SOURCE     :", "SYNTHETIC (generated now — settles and puts are invented)" if _syn
                      else f"real cache, pulled {payload.get('fetched_at')}")
print("options    :", _o.shape, "| column levels:", _o.columns.nlevels)

# Fields that came back WITH DATA (not merely present as a column)
if _o.columns.nlevels == 2:
    _fields = {}
    for f in sorted({c[1] for c in _o.columns}):
        _fields[f] = int(_o.loc[:, (slice(None), f)].notna().sum().sum())
    _rics = sorted({c[0] for c in _o.columns})
else:
    _fields = {str(_o.columns.name): int(_o.notna().sum().sum())}
    _rics = list(_o.columns)
print("fields     :", _fields, " <- non-null cell counts, 0 means the field is absent")

_cp = [parse_option_ric(r)["cp"] for r in _rics if parse_option_ric(r)]
print("rights     : C =", _cp.count("C"), "| P =", _cp.count("P"), f"(of {len(_rics)} RICs)")
print("sample RIC :", _rics[0], "->", parse_option_ric(_rics[0]))

## 1 · The raw LSEG shape

What `get_history` actually returns and what the pickle stores: a date index against
`(RIC, field)` MultiIndex columns. Most candidate RICs never existed — they simply are not here.

In [2]:
stock, options = payload["stock"], payload["options"]
print("stock:", stock.shape, "| columns:", list(stock.columns))
print("options:", options.shape, "| column index levels:", options.columns.nlevels)
options.iloc[:5, :6]

stock: (60, 4) | columns: ['OPEN_PRC', 'HIGH_1', 'LOW_1', 'TRDPRC_1']
options: (60, 1012) | column index levels: 2


RIC        UUUUF122600550.U^F26 UUUUR122600550.U^R26 UUUUF122600600.U^F26  \
Field                    SETTLE               SETTLE             TRDPRC_1   
2026-06-08               2.6430               0.1387                  NaN   
2026-06-09               2.7098               0.1173                  NaN   
2026-06-10               2.6169               0.0977                  NaN   
2026-06-11               2.3509               0.0739                  NaN   
2026-06-12               2.2345               0.0764               1.7463   

RIC                UUUUR122600600.U^R26          
Field       SETTLE             TRDPRC_1  SETTLE  
2026-06-08  2.1876                  NaN  0.1832  
2026-06-09  2.2474                  NaN  0.1549  
2026-06-10  2.1483                  NaN  0.1290  
2026-06-11  1.8746                  NaN  0.0976  
2026-06-12  1.7591                0.106  0.1010

## 2 · Tidy long table — one row per (date, ric, field) observation

Parsed RICs, spot joined per day, moneyness computed. Schema: [SYSTEM-SPEC](../docs/SYSTEM-SPEC.md) §7.1.

In [3]:
tidy = attach_underlying(flatten_lseg_options(options), stock)
print(
    f"{len(tidy):,} observations | {tidy['ric'].nunique():,} contracts | "
    f"{tidy['date'].min().date()} -> {tidy['date'].max().date()}"
)
print("observations by field:", tidy["field"].value_counts().to_dict())
tidy.head()

21,389 observations | 544 contracts | 2026-06-08 -> 2026-08-28
observations by field: {'SETTLE': 18462, 'TRDPRC_1': 2927}


,date,ric,field,value,root,cp,expiry,strike,month_code,dte,spot,moneyness
0,2026-06-08,UUUUF122600550.U^F26,SETTLE,2.6430,UUUU,C,2026-06-12,5.5,F,4,8.004346,0.687127
1,2026-06-09,UUUUF122600550.U^F26,SETTLE,2.7098,UUUU,C,2026-06-12,5.5,F,3,8.092524,0.679640
2,2026-06-10,UUUUF122600550.U^F26,SETTLE,2.6169,UUUU,C,2026-06-12,5.5,F,2,8.019258,0.685849
3,2026-06-11,UUUUF122600550.U^F26,SETTLE,2.3509,UUUU,C,2026-06-12,5.5,F,1,7.777035,0.707210
4,2026-06-12,UUUUF122600550.U^F26,SETTLE,2.2345,UUUU,C,2026-06-12,5.5,F,0,7.658083,0.718195


## 3 · Wide table — the app's working frame

One row per (date, ric), `MARK` and `TRDPRC_1` side by side ([SYSTEM-SPEC](../docs/SYSTEM-SPEC.md) §7.2).
This is the frozen interface everything downstream — and Assignment 1.2 — consumes.

In [4]:
wide = pivot_trade_settle(tidy)
mark_only = wide["has_mark"] & ~wide["has_trade"]
both_mask = wide["has_mark"] & wide["has_trade"]
print(
    f"{len(wide):,} (date, ric) rows | mark-only {mark_only.mean():.0%} | "
    f"both {both_mask.mean():.0%} | trade-only {(wide['has_trade'] & ~wide['has_mark']).mean():.0%}"
)
wide.head()

18,462 (date, ric) rows | settle-only 84% | both 16% | trade-only 0%


,date,ric,root,cp,expiry,strike,dte,spot,moneyness,SETTLE,TRDPRC_1,has_trade,has_settle,abs_diff,rel_diff
0,2026-06-08,UUUUF122600550.U^F26,UUUU,C,2026-06-12,5.5,4,8.004346,0.687127,2.6430,NaN,False,True,NaN,NaN
1,2026-06-08,UUUUF122600600.U^F26,UUUU,C,2026-06-12,6.0,4,8.004346,0.749593,2.1876,NaN,False,True,NaN,NaN
2,2026-06-08,UUUUF122600650.U^F26,UUUU,C,2026-06-12,6.5,4,8.004346,0.812059,1.7411,1.7209,True,True,0.0202,0.011602
3,2026-06-08,UUUUF122600700.U^F26,UUUU,C,2026-06-12,7.0,4,8.004346,0.874525,1.3044,NaN,False,True,NaN,NaN
4,2026-06-08,UUUUF122600750.U^F26,UUUU,C,2026-06-12,7.5,4,8.004346,0.936991,0.8786,0.8620,True,True,0.0166,0.018894


## 4 · Sparsity through time

The teaching point in one line per day: settles exist on most listed series, prints on few.

In [5]:
by_date = (
    wide.assign(mark_no_print=wide["has_mark"] & ~wide["has_trade"])
    .groupby("date")
    .agg(
        series=("ric", "nunique"),
        pct_mark_no_print=("mark_no_print", "mean"),
        prints=("has_trade", "sum"),
    )
    .reset_index()
)
display(by_date.tail(10))
px.line(
    by_date,
    x="date",
    y="pct_mark_no_print",
    title="Share of listed series with a mark and no print, by day",
    labels={"pct_mark_no_print": "mark & no trade"},
).update_yaxes(tickformat=".0%")

,date,series,pct_settle_no_print,prints
50,2026-08-17,204,0.843137,32
51,2026-08-18,196,0.867347,26
52,2026-08-19,196,0.841837,31
53,2026-08-20,204,0.808824,39
54,2026-08-21,204,0.779412,45
55,2026-08-24,180,0.872222,23
56,2026-08-25,180,0.861111,25
57,2026-08-26,180,0.822222,32
58,2026-08-27,180,0.822222,32
59,2026-08-28,180,0.822222,32


## 5 · When both exist: how far is the mark from the print?

`abs_diff = |MARK − TRDPRC_1|` on rows that have both. The median is one of the two
numbers the page must show (FR-6).

In [6]:
both = wide.dropna(subset=["TRDPRC_1", "MARK"])
print(both["abs_diff"].describe().round(4))
px.histogram(both, x="abs_diff", nbins=40, title="|MARK − TRDPRC_1| where both exist ($)")

count    2927.0000
mean        0.0573
std         0.0650
min         0.0000
25%         0.0158
50%         0.0366
75%         0.0728
max         0.6538
Name: abs_diff, dtype: float64


## 6 · One as-of date, up close

Feedstock for the FR-6 numbers and your three sentences (FR-7). Figures come from the
package's own builders — consume, don't fork.

In [7]:
asof = wide["date"].max()
sl = wide[wide["date"] == asof]
print("as-of:", asof.date())
summarize_sparsity(sl)

as-of: 2026-08-28


{'n_quotes': 180,
 'n_trade_only': 0,
 'n_settle_only': 148,
 'n_both': 32,
 'pct_settle_no_trade': 82.22222222222221,
 'median_abs_diff': 0.03185000000000002,
 'median_rel_diff_pct': 3.8297870064341764,
 'n_dates': 1,
 'n_series': 180}

In [8]:
from options_surface_lab.option_surface_plot import (
    coverage_heatmap,
    price_surface_figure,
    settle_vs_trade_figure,
)

price_surface_figure(wide, asof, cp="C", ticker=payload["ticker"])

### 6a · Moneyness (K / S) — making two as-of dates comparable (FR-10)

A strike is a fixed dollar amount; the option it describes is not. UUUU moved across the
12-week window, so the $12.50 call is out of the money one week and at the money the next.
Read two as-of dates on a raw-K axis and you are comparing two different things that happen
to share a label.

**Moneyness rebases the axis to spot** — `K / S`, already on the wide table
(`attach_underlying`). 1.00 is the money on every date, so the dense near-the-money band
sits still and the surfaces stack.

The toggle is a *ruler*, not a transform: the same points, the same interpolated sheet,
re-measured. Within one as-of date the spot is a single number, so the map is affine and the
sheet is rescaled rather than re-triangulated (`option_surface_plot._sheet_x`). A date with
no underlying close has no K / S, and those points render as holes rather than as strikes
mislabelled in ratio units (AD-9).

In [ ]:
# The same strike, walked through the window: the dollar label is fixed, the option is not.
spot_by_date = wide.groupby(wide["date"].dt.normalize())["spot"].first()
K = float(wide["strike"].median())

pd.DataFrame({"spot": spot_by_date.round(2), f"K={K:.2f}  in K / S": (K / spot_by_date).round(3)}).iloc[::8]

In [ ]:
# And the other way round: the near-the-money band is fixed in K / S and wanders in K.
# This band is where the prints actually are, so it is the part of the cloud worth comparing.
near = wide[(wide["moneyness"] - 1).abs() < 0.05]
(
    near.groupby(near["date"].dt.normalize())["strike"]
    .agg(atm_low="min", atm_high="max")
    .assign(spot=spot_by_date.round(2))
    .iloc[::8]
)

In [ ]:
# The same as-of date under both rulers. Everything else must be identical — the trace
# names, the colours, the symbols and the prices — or the toggle is doing more than it says
# (pinned by tests/test_app_figures.py::test_switching_the_axis_preserves_every_series_and_its_identity).
price_surface_figure(wide, asof, cp="C", ticker=payload["ticker"], x_mode="moneyness")

In [9]:
settle_vs_trade_figure(wide, asof, ticker=payload["ticker"])

In [10]:
coverage_heatmap(wide, asof, cp="C", field="MARK")

In [11]:
coverage_heatmap(wide, asof, cp="C", field="TRDPRC_1")

Rows to stare at — a mark, but never printed. For any of these: what would you actually
get filled at? (You don't know. That's the point, and next week's assignment.)

In [12]:
sl[sl["has_mark"] & ~sl["has_trade"]].sort_values(["cp", "strike"]).head(15)

,date,ric,root,cp,expiry,strike,dte,spot,moneyness,SETTLE,TRDPRC_1,has_trade,has_settle,abs_diff,rel_diff
18282,2026-08-28,UUUUH282600200.U^H26,UUUU,C,2026-08-28,2.0,0,5.311174,0.376565,3.3179,NaN,False,True,NaN,NaN
18300,2026-08-28,UUUUI042600200.U^I26,UUUU,C,2026-09-04,2.0,7,5.311174,0.376565,3.3289,NaN,False,True,NaN,NaN
18318,2026-08-28,UUUUI112600200.U^I26,UUUU,C,2026-09-11,2.0,14,5.311174,0.376565,3.3363,NaN,False,True,NaN,NaN
18283,2026-08-28,UUUUH282600250.U^H26,UUUU,C,2026-08-28,2.5,0,5.311174,0.470706,2.8249,NaN,False,True,NaN,NaN
18301,2026-08-28,UUUUI042600250.U^I26,UUUU,C,2026-09-04,2.5,7,5.311174,0.470706,2.8475,NaN,False,True,NaN,NaN
18319,2026-08-28,UUUUI112600250.U^I26,UUUU,C,2026-09-11,2.5,14,5.311174,0.470706,2.8625,NaN,False,True,NaN,NaN
18284,2026-08-28,UUUUH282600300.U^H26,UUUU,C,2026-08-28,3.0,0,5.311174,0.564847,2.3358,NaN,False,True,NaN,NaN
18302,2026-08-28,UUUUI042600300.U^I26,UUUU,C,2026-09-04,3.0,7,5.311174,0.564847,2.3762,NaN,False,True,NaN,NaN
18320,2026-08-28,UUUUI112600300.U^I26,UUUU,C,2026-09-11,3.0,14,5.311174,0.564847,2.4031,NaN,False,True,NaN,NaN
18285,2026-08-28,UUUUH282600350.U^H26,UUUU,C,2026-08-28,3.5,0,5.311174,0.658988,1.8514,NaN,False,True,NaN,NaN


## 8 · Checkpoint exhibits

Live material for the Class 2 conversation ([DEMO-SCRIPT](../docs/DEMO-SCRIPT.md)).

**Exhibit A — instructor question #1:** the README's printed example RIC has 10 digits
where its own Appendix A grammar (`{M}{DD}{YY}{SSSSS}`) gives 9. Run the cell: the printed
string fails the parser; the grammar-correct form works. Typo, or a venue variant we should
tolerate?

**Exhibit B** is the settled-but-never-printed table above (§6): *what fill would you
actually get on those?* (Unanswerable this week — that's Assignment 1.2.)

In [13]:
from options_surface_lab.option_surface_utils import parse_option_ric

readme_example = "UUUUA1502601250.U^A26"  # as printed in the README (10 digits after the month code)
grammar_form = "UUUUA152601250.U^A26"     # {M}{DD}{YY}{SSSSS} = 9 digits, per Appendix A

print("README's printed example parses to:", parse_option_ric(readme_example))
print("Grammar-correct 9-digit form parses to:")
parse_option_ric(grammar_form)

README's printed example parses to: None
Grammar-correct 9-digit form parses to:


{'ric': 'UUUUA152601250.U^A26',
 'root': 'UUUU',
 'cp': 'C',
 'expiry': datetime.date(2026, 1, 15),
 'strike': 12.5,
 'month_code': 'A'}

## 9 — SETTLE field diagnostic (T-27)

**Status after the 2026-08-30 pull:**

- **Puts: solved.** The expired-contract suffix takes the **call** month letter for both
  rights — `UUUUR122601100.U^F26` (put letter `R` in the body, call letter `F` in the
  suffix). The README's "repeats the month letter" form (`^R26`) returns nothing. 146 puts
  recovered; this is now the default in `build_option_ric()`. **Contradicts the README —
  worth raising with the instructor.**
- **Settle: closed 2026-08-30 — there is none.** Read §10/§10a/§10b below for the finding and
  the evidence; this bullet records where it stood mid-investigation. `SETTLE` was present as
  a column with 0 non-null cells across 294 series. The in-pull probe answered nothing: it
  requested each candidate field *alone*, and an absent field raises `LDError` rather than
  returning empty — so all seven came back `error`, including `SETTLE` itself, which the main
  pull had requested without error. Pairing each candidate with `TRDPRC_1` fixed the probe and
  produced the real answer: no settlement price exists for these instruments, and the mark
  slot is filled by `MID_PRICE` (T-32/T-34).

Cell B now pairs every candidate with `TRDPRC_1` (known to return data for these RICs) so the
request stays valid and an empty column genuinely means "field absent".

In [1]:
# A — offline: what the accidental pull actually contains
import pickle
from pathlib import Path

art = Path("..") / "option_pipeline_data.trdprc-only.pkl"
ev = pickle.load(art.open("rb"))
o = ev["options"]

print("fetched_at :", ev["fetched_at"], "| synthetic:", ev["synthetic"])
print("options    :", o.shape, "| column levels:", o.columns.nlevels, "| names:", o.columns.names)
print("non-null   :", int(o.notna().sum().sum()), "of", o.size)

# A single-level index named TRDPRC_1 is the tell: SETTLE came back all-NaN for every RIC
# and acquisition's dropna(how="all", axis=1) removed it, collapsing the MultiIndex.
ev_tidy = attach_underlying(flatten_lseg_options(o), ev["stock"])
ev_wide = pivot_trade_settle(ev_tidy)
print("\nhas_mark   any:", bool(ev_wide["has_mark"].any()))
print("has_trade  any:", bool(ev_wide["has_trade"].any()))
print("rights     :", ev_wide["cp"].value_counts().to_dict())
print("expiries   :", sorted(ev_wide["expiry"].dropna().unique().astype(str)))

SyntaxError: unterminated string literal (detected at line 17) (1663140997.py, line 17)

In [2]:
# B — LIVE PROBE, read-only. Requires LSEG Workspace. Writes nothing, touches no cache.
# Pairs each candidate with TRDPRC_1 so an absent field returns EMPTY instead of raising.
RUN_PROBE = False

if RUN_PROBE:
    import pandas as pd
    from options_surface_lab.options_surface_app import probe_mark_fields

    result = probe_mark_fields(n_rics=5)
    print("window :", result["window"])
    print("sample :", result["sample_rics"][:3], "...")
    print("winner :", result["winner"] or "none — no field carries the mark")
    display(pd.DataFrame(
        [{"field": k, "non_null_cells": v} for k, v in result["results"].items()]
    ))
else:
    print("RUN_PROBE is False — flip it with Workspace running to settle the settle question.")

SyntaxError: unterminated string literal (detected at line 17) (2498679097.py, line 17)

**Record the answer here before re-pulling (feeds T-6/T-7 and PRD OQ-4):**

- Field that carries the exchange mark for expired US equity options: _____
- Do put RICs return data at all, or is the put month-code mapping wrong? _____
- If no field carries settle for expired contracts, that becomes the FR-7 finding and
  FR-2 needs a scope conversation with the PO.

## 7 · Scratchpad

Poke below. Keep transforms out of here — anything worth keeping graduates into
`option_surface_utils.py` with a test.

---

# 10 — Checkpoint exhibit: there is no SETTLE for these contracts

**The claim:** the assignment says compare `SETTLE` to `TRDPRC_1`. For *expired US listed
equity options*, LSEG does not carry a settle at all — under any name. The thesis survives,
but the mark has to come from somewhere else.

**Three things to show, in this order.** Run the cell below; it uses the live session if
Workspace is up and the captured evidence file if it isn't, so it works either way.

1. **Ask what these contracts carry.** Request the history with *no* field list — whatever
   comes back is the truth. 22 fields, no settle among them; checked on 14 RICs spread over
   7 expiries and both rights, which return **one identical field set**.
2. **Rule out our own mistake.** Seven settle-ish names across **all 294 series × 53 days =
   15,582 contract-days**, each paired with `TRDPRC_1` so an absent field returns empty
   instead of raising — every one is zero, and every RIC returned data on the paired field.
   Then the control: `SETTLE` returns real values for `CLc1` (a future) *in the same
   session*, returns nothing for the underlying equity, and errors with *"No successful
   response"* when asked for alone on an option. So the field works — these instruments just
   don't have one.

   *Scope, if he pushes:* this is measured for every expired UUUU contract in the 12-week
   window. Other underlyings and other windows are not measured — the step to "US equity
   options generally" is an inference from the instrument class plus the `CLc1` control.
3. **Show the thesis holds anyway.** The quoted mark exists in contracts that never printed:
   **121 contract-days carry a mark with no trade behind it.**

**The question for the instructor:** was that the intended discovery — and if so, should the
mark be the quoted mid (`MID_PRICE`) or the model value (`THEO_VALUE`)?

In [ ]:
import json
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

EV = json.loads((Path.cwd() / "settle_field_evidence.json").read_text())

print(f"Window {EV['window'][0]} to {EV['window'][1]}   |   {EV['n_sample_rics']} contracts, "
      f"{EV['contract_days']} contract-days\n")

# 1 — what these contracts actually carry
FL = EV["field_list"]
print("1. FIELDS THESE RICs RETURN (asked with no field list):")
print("  ", ", ".join(FL["fields"]))
print(f"\n   checked on {FL['n_rics_checked']} RICs across 7 expiries and both rights")
print(f"   distinct field sets: {FL['distinct_field_sets']}   SETTLE in any: {FL['settle_in_any']}\n")

# 2 — not our mistake
SC = EV["settle_candidates"]
print(f"2. EVERY SETTLE-ISH NAME over {SC['_scope']}:")
print("  ", SC["counts"], "<- all zero")
print(f"   ({SC['method']})\n")
print("   CONTROL — same session, same call:")
for tag, row in EV["control"].items():
    got = row.get("settle_values", row.get("error"))
    print(f"     {row['universe']:<22} {tag:<26} SETTLE values: {got}")
print(f"     SETTLE alone on an option -> {EV['settle_alone_on_option']}\n")

# 3 — the thesis holds on a different mark
cov = EV["coverage"]; tot = EV["contract_days"]
tbl = pd.DataFrame([{"field": f, "contract_days_with_a_value": n,
                     "coverage": f"{100*n/tot:.1f}%"} for f, n in cov.items()])
print("3. WHAT IS THERE — coverage vs the last trade:")
display(tbl)
for f, d in EV["mark_without_print"].items():
    print(f"   {f}: {d['both']} with a trade, {d['mark_only']} MARK WITH NO PRINT")

TRADE, MARK = "#ff0055", "#00ffcc"
fig = go.Figure(go.Bar(
    x=[100 * cov[f] / tot for f in cov], y=list(cov), orientation="h",
    marker_color=[TRADE if f == "TRDPRC_1" else MARK for f in cov],
    text=[f"{100*cov[f]/tot:.1f}%" for f in cov], textposition="outside"))
fig.update_layout(
    title="Expired UUUU options — a mark exists where no trade printed",
    xaxis_title="% of contract-days with a value", template="plotly_dark",
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117", height=420,
    yaxis={"categoryorder": "total ascending"}, margin={"r": 80})
fig.add_annotation(x=100*cov["TRDPRC_1"]/tot, y="TRDPRC_1", text="last trade (magenta)",
                   showarrow=True, arrowhead=2, ax=70, ay=30, font={"color": TRADE})
fig.show()

### 10a — The two documented alternatives, and why they fail

Searching LSEG's docs turns up two candidate routes. Both were tested; the results are in
`settle_field_evidence.json` under `tr_namespace_probe`.

| route | result |
|---|---|
| `TR.SETTLEMENTPRICE` | 1 value on an expired option, **15 on `CLc1`**. It is a *futures* settlement field. |
| `TR.OptionSettlementPrice` | `LDError` — the field does not exist. |
| `TR.CLOSEPRICE` | Returns data, but after deduplication it is **identical to `TRDPRC_1` in 356 of 356 overlapping observations**. It is the last trade re-served. |

**The `TR.CLOSEPRICE` trap.** It returns 55 rows for a contract that traded on 3 days — but
those rows are *padded repeats*: one date recurs ~50 times carrying the same value. Counting
rows suggests rich data; deduplicating on `(ric, date)` collapses it to the trade days plus
one stale prior close. This is precisely what README line 143 warns about — *"Do not use
`CLOSE` and `SETTLE` as synonyms without checking the field list."*

**Independent corroboration.** [LSEG's own article on expired
options](https://developers.lseg.com/en/article-catalog/article/finding-expired-options-and-backtesting-a-short-iron-condor-stra)
backtests a short iron condor using `BID`, `ASK` and `TRDPRC_1` — no settle field anywhere.
It reaches for `TR.PriceClose` only for the *underlying*. It also states the expired suffix is
the **expiration** month code and year, which is what we measured (`^F26` for both rights).

### 10b — Why there is no settle to find

The field is missing from LSEG because **the thing does not exist in the market.** There is no
official close for the NBBO, and no settlement price, distributed by the options exchanges, by
OPRA, or by the OCC for US listed equity options.

So it is not a data-access problem and there is no better endpoint. **Every end-of-day option
mark is derived by whoever needs one**, and the industry uses two approaches:

| method | what it is | already in our data |
|---|---|---|
| **Mechanical** | closing bid/ask midpoint, or the last print | `MID_PRICE`, `BID`, `ASK` |
| **Theoretical** | snapshot quotes near the close, fit a pricing surface, read marks off it | `THEO_VALUE` |

Don't confuse this with the OCC's *expiration* settlement — that is a VWAP of the
**underlying** over the last 30 minutes, used to decide exercise. It is not a daily mark for
each option series.

**Why this matters for AD-9.** The "theoretical" method is exactly what this app's interpolated
sheet does. Using `THEO_VALUE` as the mark *and* drawing an interpolated sheet would be a model
on top of a model. `MID_PRICE` keeps the mark market-derived, so the sheet stays visibly the
assumption we are imposing — which is the point the page is making.

**The assignment's premise, restated honestly:** "SETTLE vs TRDPRC_1" cannot be done as
written, because SETTLE does not exist for these instruments. What *can* be shown — and is
arguably the better lesson — is **quoted mark vs last print**: the mark exists in contracts
that never traded, and every mark you have ever seen was somebody's derivation.